<a href="https://colab.research.google.com/github/shin-noda/deep-learning-with-python/blob/main/Chapter15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Training a Shakespeare language model

In [ ]:
import keras

filename = keras.utils.get_file(
    origin=(
        "https://storage.googleapis.com/download.tensorflow.org/"
        "data/shakespeare.txt"
    ),
)

shakespeare = open(filename, "r").read()

In [ ]:
print(shakespeare[:250])

In [ ]:
import tensorflow as tf

# The chunk size we will use during training.
# We only train on sequences of 100 characters at a time.
sequence_length = 100

def split_input(input, sequence_length):
    for i in range(0, len(input), sequence_length):
        yield input[i : i + sequence_length]

features = list(split_input(shakespeare[:-1], sequence_length))
labels = list(split_input(shakespeare[1:], sequence_length))
dataset = tf.data.Dataset.from_tensor_slices((features, labels))

In [ ]:
x, y = next(dataset.as_numpy_iterator())
x[:50], y[:50]

In [ ]:
from keras import layers

tokenizer = layers.TextVectorization(
    standardize=None,
    split="character",
    output_sequence_length=sequence_length,
)

tokenizer.adapt(dataset.map(lambda text, labels: text))

In [ ]:
vocabulary_size = tokenizer.vocabulary_size()

In [ ]:
dataset = dataset.map(
    lambda features, labels: (tokenizer(features), tokenizer(labels)),
    num_parallel_calls=8,
)

training_data = dataset.shuffle(10_000).batch(64).cache()

In [ ]:
embedding_dim = 256
hidden_dim = 1024

inputs = layers.Input(shape=(sequence_length,), dtype="int", name="token_ids")

x = layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x = layers.GRU(hidden_dim, return_sequences=True)(x)
x = layers.Dropout(0.1)(x)

# Outputs a probability distribution over all potential toekns in our vocabulary
outputs = layers.Dense(vocabulary_size, activation="softmax")(x)

model = keras.Model(inputs, outputs)

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)

model.fit(training_data, epochs=20)

In [ ]:
# Creates a model that receives and outputs the RNN state
inputs = keras.Input(shape=(1,), dtype="int", name="token_ids")
input_state = keras.Input(shape=(hidden_dim,), name="state")

x = layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x, output_state = layers.GRU(hidden_dim, return_state=True)(
    x, initial_state=input_state
)

outputs = layers.Dense(vocabulary_size, activation="softmax")(x)

generation_model = keras.Model(
    inputs=(inputs, input_state),
    outputs=(outputs, output_state),
)

# Copies the parameters from the original model
generation_model.set_weights(model.get_weights())

In [ ]:
tokens = tokenizer.get_vocabulary()
token_ids = range(vocabulary_size)
char_to_id = dict(zip(tokens, token_ids))
id_to_char = dict(zip(token_ids, tokens))

prompt = """
KING RICHARD III:
"""

In [ ]:
input_ids = [char_to_id[c] for c in prompt]
state = keras.ops.zeros(shape=(1, hidden_dim))

for token_id in input_ids:
    inputs = keras.ops.expand_dims([token_id], axis=0)

    # Feeds the prompt character by character to update state
    predictions, state = generation_model.predict((inputs, state), verbose=0)

In [ ]:
import numpy as np

generated_ids = []
max_length = 250

# Generates characters one by one, computing a new state each iteration
for i in range(max_length):
    # The next character is the output index with the highest probability.
    next_char = int(np.argmax(predictions, axis=-1)[0])
    generated_ids.append(next_char)
    inputs = keras.ops.expand_dims([next_char], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)

In [ ]:
output = "".join([id_to_char[token_id] for token_id in generated_ids])
print(prompt + output)

In [ ]:
# English-to-Spanish translation
import pathlib

zip_path = keras.utils.get_file(
    origin=(
        "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
    ),
    fname="spa-eng",
    extract=True,
)

text_path = pathlib.Path(zip_path) / "spa-eng" / "spa.txt"

In [ ]:
with open(text_path) as f:
    lines = f.read().split("\n")[:-1]

text_pairs = []
for line in lines:
    english, spanish = line.split("\t")
    spanish = "[start]" + spanish + "[end]"
    text_pairs.append((english, spanish))

In [ ]:
import random
random.choice(text_pairs)

In [ ]:
import random

random.shuffle(text_pairs)
val_samples = int(0.15 * len(text_pairs))

train_samples = len(text_pairs) - 2 * val_samples
train_pairs = text_pairs[:train_samples]

val_pairs = text_pairs[train_samples : train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples :]

In [ ]:
import string
import re

strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)

    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", ""
    )

vocab_size = 15000
sequence_length = 20

english_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)

spanish_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)

train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]

english_tokenizer.adapt(train_english_texts)
spanish_tokenizer.adapt(train_spanish_texts)

In [ ]:
batch_size = 64

def format_dataset(eng, spa):
    eng = english_tokenizer(eng)
    spa = spanish_tokenizer(spa)

    features = {"english": eng, "spanish": spa[:, :-1]}
    labels = spa[:, 1:]
    sample_weights = labels != 0

    return features, labels, sample_weights


def make_dataset(pairs):
    eng_texts, spa_texts = zip(*pairs)

    eng_texts = list(eng_texts)
    spa_texts = list(spa_texts)

    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)

    return dataset.shuffle(2048).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

In [ ]:
inputs, targets, sample_weights = next(iter(train_ds))
print(inputs["english"].shape)
print(inputs["spanish"].shape)
print(targets.shape)
print(sample_weights.shape)

In [ ]:
# Sequence-to-sequence learning with RNNs
inputs = keras.Input(shape=(sequence_length,), dtype="int32")

x = layers.Embedding(input_dim=vocab_size, output_dim=128)(inputs)
x = layers.LSTM(32, return_sequences=True)(x)

outputs = layers.Dense(vocab_size, activation="softmax")(x)
model = keras.Model(inputs, outputs)

In [ ]:
embed_dim = 256
hidden_dim = 1024

source = keras.Input(shape=(None,), dtype="int32", name="english")

x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(source)

rnn_layer = layers.GRU(hidden_dim)
rnn_layer = layers.Bidirectional(rnn_layer, merge_mode="sum")

encoder_output = rnn_layer(x)

In [ ]:
target = keras.Input(shape=(None,), dtype="int32", name="spanish")

x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(target)

rnn_layer = layers.GRU(hidden_dim, return_sequences=True)

x = rnn_layer(x, initial_state=encoder_output)
x = layers.Dropout(0.5)(x)

# Predicts the next word of the translation, given the current word
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
seq2seq_rnn = keras.Model([source, target], target_predictions)

In [ ]:
seq2seq_rnn.summary()

In [ ]:
seq2seq_rnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"]
)

seq2seq_rnn.fit(train_ds, epochs=15, validation_data=val_ds)

In [ ]:
import numpy as np

spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"

    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = seq2seq_rnn.predict(inputs, verbose=0)

        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token

        if sampled_token == "[end]":
            break

    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]

for _ in range(5):
    input_sentence = random.choice(test_eng_texts)

    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

In [ ]:
# The Transformer architecture

In [ ]:
# # Transposes
# np.einsum("ij->ji")

# # matmul
# np.einsum("ij,jk->ik")

# # matmuls a list of matrices against a single matrix
# np.einsum("hij,jk->hik")

# # Dot-product
# np.einsum("i,i->")

# # Element-wise multiplication
# np.einsum("ijk,ijk->ijk")

# # Element-wise multiplies and sums everything.
# np.einsum("ijk,ijk->")

In [ ]:
def dot_product_attention(target, source):
    # Takes the dot-product between all target and source vectors,
    # where b = batch_size, t = target length, s = source length, and d = vector size
    scores = np.einsum("btd,bsd->bts", target, source)
    scores = softmax(scores, axis=-1)

    # Computes a wieghted sum of all source vectors for each target vector
    return np.einsum("bts,bsd->btd", scores, source)

In [ ]:
def parameterized_attention(query, key, value):
    query = query_dense(query)
    key = key_dense(key)
    value = value_dense(value)

    scores = np.einsum("btd,bsd->bts", query, key)
    scores = softmax(scores, axis=-1)

    outputs = np.einsum("bts,bsd->btd", scores, value)

    return output_dense(outputs)

In [ ]:
def multi_head_attention(query, key, value):
    head_outputs = []

    for i in range(num_heads):
        query = query_dense[i](query)
        key = key_dense[i](key)
        value = value_dense[i](value)

        scores = np.einsum("btd,bsd->bts", target, source)
        scores = softmax(scores / math.sqrt(head_dim), axis=-1)

        head_output = np.einsum("bts,bsd->btd", scores, source)
        head_outputs.append(head_output)

    outputs = ops.concatenate(head_outputs, axis=-1)

    return output_dense(outputs)

In [ ]:
# multi_head_attention = keras.layers.MultiHeadAttention(
#     num_heads=num_heads,
#     head_dim=head_dim,
# )

# multi_head_attention(query=target, key=source, value=source)

In [ ]:
class TransformerEncoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads

        # Self-attention layers
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()

        # Feedforward layers
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()


    def call(self, source, source_mask):
        # Self-attention computation
        residual = x = source
        mask = source_mask[:, None, :]

        x = self.self_attention(query=x, key=x, value=x, attention_mask=mask)
        x = x + residual
        x = self.self_attention_layernorm(x)

        # Feedforward computation
        residual = x

        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)

        return x

In [ ]:
class TransformerDecoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads

        # Self-attention layers
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()

        # Cross-attention layers
        self.cross_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.cross_attention_layernorm = layers.LayerNormalization()

        # Feedforward layers
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()


    def call(self, target, source, source_mask):
        # Self-attention computation
        residual = x = target

        x = self.self_attention(query=x, key=x, value=x, use_causal_mask=True)
        x = x + residual
        x = self.self_attention_layernorm(x)

        # Cross-attention computation
        residual = x
        mask = source_mask[:, None, :]

        x = self.cross_attention(
            query=x, key=source, value=source, attention_mask=mask
        )

        x = x + residual
        x = self.cross_attention_layernorm(x)

        # Feedforward computation
        residual = x

        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)

        return x

In [ ]:
hidden_dim = 256
intermediate_dim = 2048
num_heads = 8

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

target = keras.Input(shape=(None,), dtype="int32", name="spanish")

x = layers.Embedding(vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)

x = layers.Dropout(0.5)(x)

target_predictions = layers.Dense(vocab_size, activation="softmax")(x)

transformer = keras.Model([source, target], target_predictions)

In [ ]:
transformer.summary()

In [ ]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"]
)

transformer.fit(train_ds, epochs=15, validation_data=val_ds)

In [ ]:
# Embedding positional information

In [ ]:
import keras
from keras import ops

class PositionalEmbedding(keras.Layer):
    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        self.token_embeddings = layers.Embedding(input_dim, output_dim)
        self.position_embeddings = layers.Embedding(sequence_length, output_dim)


    def call(self, inputs):
        # Computes incrementing positions [0, 1, 2...] for each sequence in the batch
        positions = ops.cumsum(ops.ones_like(inputs), axis=-1) - 1
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)

        return embedded_tokens + embedded_positions

In [ ]:
hidden_dim = 256
intermediate_dim = 2056
num_heads = 8

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

target = keras.Input(shape=(None,), dtype="int32", name="spanish")

x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)

x = layers.Dropout(0.5)(x)

target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions)

In [ ]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"]
)

transformer.fit(train_ds, epochs=30, validation_data=val_ds)

In [ ]:
import numpy as np

spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"

    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        tokenized_target_sentence = tokenized_target_sentence[:, :-1]

        inputs = [tokenized_input_sentence, tokenized_target_sentence]

        next_token_predictions = transformer.predict(inputs, verbose=0)

        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token

        if sampled_token == "[end]":
            break

    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]

for _ in range(5):
    input_sentence = random.choice(test_eng_texts)

    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

In [ ]:
import keras_hub

tokenizer = keras_hub.models.Tokenizer.from_preset("roberta_base_en")
backbone = keras_hub.models.Backbone.from_preset("roberta_base_en")

In [ ]:
tokenizer("The quick brown fox")

In [ ]:
backbone.summary()

In [ ]:
import os, pathlib, shutil, random, keras

zip_path = keras.utils.get_file(
    origin="https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz",
    fname="imdb",
    extract=True,
)

imdb_extract_dir = pathlib.Path(zip_path) / "aclImdb"

for path in imdb_extract_dir.glob("*/*"):
    if path.is_dir():
        print(path)

train_dir = pathlib.Path("imdb_train")
test_dir = pathlib.Path("imdb_test")
val_dir = pathlib.Path("imdb_val")

In [ ]:
# # Moves the test data unaltered
# shutil.copytree(imdb_extract_dir / "test", test_dir)

# # Splits the training data into a train set and a validation set
# val_percentage = 0.2

# for category in ("neg", "pos"):
#     src_dir = imdb_extract_dir / "train" / category
#     src_files = os.listdir(src_dir)
#     random.Random(1337).shuffle(src_files)
#     num_val_samples = int(len(src_files) * val_percentage)

#     os.makedirs(val_dir / category)

#     for file in src_files[:num_val_samples]:
#         shutil.copy(src_dir / file, val_dir / category / file)

#     os.makedirs(train_dir / category)

#     for file in src_files[num_val_samples:]:
#         shutil.copy(src_dir / file, train_dir / category / file)

In [ ]:
# Preprocessing IMDb movie reviews
from keras.utils import text_dataset_from_directory

batch_size = 16

train_ds = text_dataset_from_directory(train_dir, batch_size=batch_size)
val_ds = text_dataset_from_directory(val_dir, batch_size=batch_size)
test_ds = text_dataset_from_directory(test_dir, batch_size=batch_size)

In [ ]:
def preprocess(text, label):
    packer = keras_hub.layers.StartEndPacker(
        sequence_length=512,
        start_value=tokenizer.start_token_id,
        end_value=tokenizer.end_token_id,
        pad_value=tokenizer.pad_token_id,
        return_padding_mask=True,
    )

    token_ids, padding_mask = packer(tokenizer(text))

    return {"token_ids": token_ids, "padding_mask": padding_mask}, label

preprocessed_train_ds = train_ds.map(preprocess)
preprocessed_val_ds = val_ds.map(preprocess)
preprocessed_test_ds = test_ds.map(preprocess)

In [ ]:
next(iter(preprocessed_train_ds))

In [ ]:
# Fine-tuning a pretrained Transformer

In [ ]:
from keras import layers

inputs = backbone.input
x = backbone(inputs)

# Uses the hidden representation of the first token
x = x[:, 0, :]

x = layers.Dropout(0.1)(x)
x = layers.Dense(768, activation="relu")(x)
x = layers.Dropout(0.1)(x)

outputs = layers.Dense(1, activation="sigmoid")(x)
classifier = keras.Model(inputs, outputs)

In [ ]:
classifier.compile(
    optimizer=keras.optimizers.Adam(5e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

classifier.fit(
    preprocessed_train_ds,
    validation_data=preprocessed_val_ds,
)

In [ ]:
classifier.evaluate(preprocessed_test_ds)